### Base Imports

In [1]:
import os
import json
import random
import pandas as pd
from collections import defaultdict

random.seed(42)

df_img = pd.read_csv("../metadata_output/metadata_images.csv")
df_q   = pd.read_csv("../metadata_output/metadata_questions.csv")

with open("../ai2d/categories.json") as f:
    categories = json.load(f)

print(f"Before filtering: {len(df_img)} images, {len(df_q)} questions")

Before filtering: 4903 images, 15501 questions


### Drop Un-usable Imgs

In [2]:
# Drop 1: images with no question file
# Drop 2: images with a question file but 0 actual questions inside
df_img = df_img[df_img["q_question_count"] > 0].reset_index(drop=True)

# Keep only questions belonging to remaining images
valid_images = set(df_img["image_name"])
df_q = df_q[df_q["image_name"].isin(valid_images)].reset_index(drop=True)

print(f"After filtering: {len(df_img)} images, {len(df_q)} questions")

After filtering: 4060 images, 15501 questions


### 01 - Stratified dataset split

In [3]:
by_category = defaultdict(list)
for img in df_img["image_name"]:
    cat = categories.get(img, "other")
    by_category[cat].append(img)

train_imgs, val_imgs, test_imgs = [], [], []

for cat, imgs in by_category.items():
    random.shuffle(imgs)
    n      = len(imgs)
    n_test = max(1, int(n * 0.15))
    n_val  = max(1, int(n * 0.15))
    test_imgs  += imgs[:n_test]
    val_imgs   += imgs[n_test:n_test + n_val]
    train_imgs += imgs[n_test + n_val:]

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

# Sanity check — no overlap
assert not set(train_imgs) & set(val_imgs), "Train/Val overlap!"
assert not set(train_imgs) & set(test_imgs), "Train/Test overlap!"
assert not set(val_imgs)   & set(test_imgs), "Val/Test overlap!"
print("✓ No overlap between splits")

Train: 2858 | Val: 601 | Test: 601
✓ No overlap between splits


In [7]:
# Verify category distribution across splits
split_map = (
    {img: "train" for img in train_imgs} |
    {img: "val"   for img in val_imgs}   |
    {img: "test"  for img in test_imgs}
)

df_img["split"] = df_img["image_name"].map(split_map)
df_img["category"] = df_img["image_name"].map(categories)

check = df_img.groupby(["category", "split"]).size().unstack(fill_value=0)
check["total"] = check.sum(axis=1)
check["train%"] = (check["train"] / check["total"] * 100).round(1)
check["val%"]   = (check["val"]   / check["total"] * 100).round(1)
check["test%"]  = (check["test"]  / check["total"] * 100).round(1)

print(check[["train", "val", "test", "train%", "val%", "test%"]])

split                      train  val  test  train%  val%  test%
category                                                        
atomStructure                 14    2     2    77.8  11.1   11.1
eclipses                      59   12    12    71.1  14.5   14.5
faultsEarthquakes             52   10    10    72.2  13.9   13.9
foodChainsWebs               431   92    92    70.1  15.0   15.0
lifeCycles                   379   80    80    70.3  14.8   14.8
moonPhaseEquinox             311   66    66    70.2  14.9   14.9
other                          2    1     1    50.0  25.0   25.0
partsOfA                     765  163   163    70.1  14.9   14.9
partsOfTheEarth               81   16    16    71.7  14.2   14.2
photosynthesisRespiration     65   13    13    71.4  14.3   14.3
rockCycle                     75   16    16    70.1  15.0   15.0
rockStrata                    57   11    11    72.2  13.9   13.9
solarSystem                   29    5     5    74.4  12.8   12.8
typesOf                  

### 02 - Annotation summary builder

In [14]:
def build_annotation_context(image_name):
    ann_path = f"../ai2d/annotations/{image_name}.json"
    if not os.path.exists(ann_path):
        return ""
    with open(ann_path) as f:
        ann = json.load(f)

    texts         = ann.get("text", {})
    relationships = ann.get("relationships", {})

    # Build label lookup: tid → "Label(X)"
    label_map = {}
    for tid, t in texts.items():
        val  = t.get("value", "").strip()
        repl = t.get("replacementText", "").strip()
        if val:
            label_map[tid] = f"{val}({repl})" if repl else val

    # Build connections
    connections = []
    title       = None

    for rel in relationships.values():
        cat    = rel.get("category", "")
        origin = rel.get("origin", "")
        dest   = rel.get("destination", "")

        origin_str = label_map.get(origin)
        dest_str   = label_map.get(dest)

        if cat == "imageTitle" and origin_str:
            title = origin_str

        elif cat == "intraObjectLabel":
            # both ends are readable text nodes
            if origin_str and dest_str:
                connections.append(f"{origin_str} labels {dest_str}")
            elif origin_str:
                connections.append(f"{origin_str} → visual region")

        elif cat == "intraObjectLinkage":
            if origin_str:
                connections.append(f"{origin_str} → visual region")

    # Build context string
    context = ""
    if title:
        context += f"Diagram title: {title}\n"
    if label_map:
        context += f"Diagram labels: {', '.join(label_map.values())}\n"
    if connections:
        context += "Connections:\n" + "\n".join(f"  {c}" for c in connections) + "\n"

    return context

### 03 - Sample builder

In [15]:
def build_samples(image_names, version="A"):
    samples  = []
    options  = ["A", "B", "C", "D"]

    for img_name in image_names:
        q_rows = df_q[df_q["image_name"] == img_name]

        ann_context = ""
        if version == "B":
            ann_context = build_annotation_context(img_name)

        for _, row in q_rows.iterrows():
            answers     = row["answer_texts"].split(" | ")
            correct_idx = int(row["correct_answer_index"])

            choices_str = "  ".join(
                f"{options[i]}) {ans}" for i, ans in enumerate(answers)
            )
            correct_letter = options[correct_idx]
            correct_text   = answers[correct_idx]

            question_str = row["question_text"]
            if row["abc_label"]:
                question_str = "[Label Question] " + question_str

            user_text = f"{ann_context}Question: {question_str}\n{choices_str}"

            samples.append({
                "image_name":    img_name,
                "image_path":    f"../ai2d/images/{img_name}",
                "category":      categories.get(img_name, "other"),
                "question_id":   row["question_id"],
                "abc_label":     bool(row["abc_label"]),
                "user_text":     user_text,
                "answer":        f"{correct_letter}) {correct_text}",
                "correct_index": correct_idx,
                "version":       version,
            })

    return samples

### 04 - Build and save all splits

In [16]:
for split_name, split_imgs in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]:
    for ver in ["A", "B"]:
        samples  = build_samples(split_imgs, version=ver)
        out_path = f"../processed_data/splits/{split_name}_v{ver}.json"
        with open(out_path, "w") as f:
            json.dump(samples, f, indent=2)
        print(f"✓ {out_path}  ({len(samples)} samples)")

✓ ../processed_data/splits/train_vA.json  (10918 samples)
✓ ../processed_data/splits/train_vB.json  (10918 samples)
✓ ../processed_data/splits/val_vA.json  (2300 samples)
✓ ../processed_data/splits/val_vB.json  (2300 samples)
✓ ../processed_data/splits/test_vA.json  (2283 samples)
✓ ../processed_data/splits/test_vB.json  (2283 samples)


### 05 -  Verify a sample from each version

In [17]:
with open("../processed_data/splits/train_vA.json") as f:
    sample_A = json.load(f)[0]

with open("../processed_data/splits/train_vB.json") as f:
    sample_B = json.load(f)[0]

print("=== Version A (image + question only) ===")
print(f"Image:    {sample_A['image_name']}")
print(f"Prompt:   {sample_A['user_text']}")
print(f"Answer:   {sample_A['answer']}")

print("\n=== Version B (image + question + annotation) ===")
print(f"Image:    {sample_B['image_name']}")
print(f"Prompt:   {sample_B['user_text']}")
print(f"Answer:   {sample_B['answer']}")

=== Version A (image + question only) ===
Image:    3020.png
Prompt:   Question: [Label Question] What is C in the diagram?
A) fall  B) style branch  C) stem  D) standard
Answer:   C) stem

=== Version B (image + question + annotation) ===
Image:    3020.png
Prompt:   Diagram labels: Standard(A), Style Branch(D), Stem(C), Fall(B)
Connections:
  Standard(A) → visual region
  Style Branch(D) → visual region
  Stem(C) → visual region
  Fall(B) → visual region
Question: [Label Question] What is C in the diagram?
A) fall  B) style branch  C) stem  D) standard
Answer:   C) stem
